### 1. TextLoader

`TextLoader` is the simplest document loader and the best place to start when learning LangChain. It reads plain text files and converts them into `Document` objects, where each document contains the actual text and metadata. In real AI engineering, this is useful for local notes, transcripts, policy documents, or source text that will later be chunked and embedded.

What to remember:
- Always check character encoding, especially for languages with accents or non-English text
- Use metadata to keep track of the source file
- If the file is large, split it into chunks before sending it to an LLM
- The loader does not replace chunking or retrieval; it only prepares the data

This is the entry point for many RAG pipelines: raw file -> `Document` -> chunking -> embeddings -> retrieval -> answer generation.

### Why DocumentLoader matters in LangChain

DocumentLoader is the ingestion layer for LLM apps. It turns raw sources like `.txt`, `.csv`, `.pdf`, or webpages into LangChain `Document` objects with:

- `page_content`: the actual text that the model reads
- `metadata`: source name, page number, URL, file path, etc.

This matters because your application is only as good as the text you feed into the model. In real AI engineering work, the main goal is not just "load data" but to load clean, relevant, and traceable text for retrieval, summarization, and question-answering.

Common AI-engineering mistakes:
- Using raw HTML or scraped text without cleaning
- Ignoring metadata needed for traceability
- Loading huge files without considering chunking and token limits
- Choosing the wrong loader for the source format

In RAG systems, the loader is usually the first step in the pipeline: source -> loader -> split -> embed -> retrieve -> answer.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI()

prompt = PromptTemplate(
    template='Write a summary for the following poem - \n {poem}',
    input_variables=['poem']
)

parser = StrOutputParser()

loader = TextLoader('D:\\LangChain\\langchain-document-loaders-main\\cricket.txt', encoding='utf-8')

docs = loader.load()
print(type(docs))
print(len(docs))
print(docs[0].page_content)
print(docs[0].metadata)
chain = prompt | model | parser
print(chain.invoke({'poem':docs[0].page_content}))

C:\Users\sidha\AppData\Local\Temp\ipykernel_22556\474713584.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\sidha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


<class 'list'>
1
Beneath the sun or floodlight's gleam,

Cricket lives like a waking dream.

A field of green, a willowed sound,

Where legends rise and tales are found.

From dusty lanes where barefoot boys,

Chase every run with shrieks of joy,

To packed arenas roaring loud,

The game unites a global crowd.

A coin is tossed, the captains stare,

As tension thickens in the air.

Bat or bowl? A choice so bold,

A story new begins, retold.

The openers walk, calm yet brave,

Each stride a wave upon the wave.

They face the ball with narrowed eyes,

As silence grips the watching skies.

The bowler runs, a rhythmic beat,

Like thunder galloping on feet.

A leather flash, a wooden crack—

The ball takes flight, then tumbles back.

A flick through square, a drive through mid,

A lofted shot the fielder missed.

A single, double, sprint for three,

The crowd erupts in ecstasy.

But not for long—the trap is set,

The spinner loops, the pitch is wet.

A sudden turn, the bat’s deceived,

And 

### CSV Loader

`CSVLoader` is useful when your data lives in tabular form but you still want to feed it into a retrieval or summarization workflow. LangChain converts each row into a `Document`, and the row values become part of the text that the model can read.

Why this matters for AI engineering:
- CSV files are common in business workflows and data pipelines
- Structured data often needs cleaning before it becomes useful to an LLM
- Row-level conversion can create noisy documents if the file contains many columns or irrelevant fields

Best practice: select only the columns that matter, clean the values, and keep the text concise so the model can extract useful answers without getting confused by raw table data.

In [3]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path='D:\LangChain\langchain-document-loaders-main\Social_Network_Ads.csv')

docs = loader.load()
    
print(len(docs))
print(docs[1])

400
page_content='User ID: 15810944
Gender: Male
Age: 35
EstimatedSalary: 20000
Purchased: 0' metadata={'source': 'D:\\LangChain\\langchain-document-loaders-main\\Social_Network_Ads.csv', 'row': 1}


<>:3: SyntaxWarning: invalid escape sequence '\L'
<>:3: SyntaxWarning: invalid escape sequence '\L'
C:\Users\sidha\AppData\Local\Temp\ipykernel_22556\3347025261.py:3: SyntaxWarning: invalid escape sequence '\L'
  loader = CSVLoader(file_path='D:\LangChain\langchain-document-loaders-main\Social_Network_Ads.csv')


### 3. PDF Loader

### 3. PDF Loader

PDFs are common in enterprise AI workflows, but they are often messy: multiple columns, scanned pages, headers, and page breaks. `PyPDFLoader` helps extract text from PDF pages into LangChain `Document` objects. In production systems, PDFs usually need additional preprocessing such as:

- page cleanup
- chunking by document sections
- metadata retention
- OCR for scanned documents

When using PDF loaders, remember that the quality of the extracted text directly affects retrieval and answer quality.

### Other popular PDF loaders in LangChain

Besides `PyPDFLoader`, there are a few common options engineers use depending on the use case:

- `UnstructuredPDFLoader`: good for messy PDFs and documents where structure matters
- `PDFPlumberLoader`: useful when you want more control over text extraction and layout
- `AzureAIDocumentIntelligenceLoader`: useful in enterprise pipelines when you want OCR and document understanding from Azure AI services

A good rule is: if the PDF is clean and text-based, use a simple loader like `PyPDFLoader`; if it is scanned or structurally complex, choose a more advanced extraction approach.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('D:\LangChain\langchain-document-loaders-main\dl-curriculum.pdf')
docs = loader.load()

print(len(docs))
print(docs[0].page_content)
print(docs[1].metadata)

### 4 DirectoryLoader

`DirectoryLoader` is the practical tool for scanning a folder and loading many files in one pass. This is highly useful when working with document collections such as research notes, PDFs, or text files in a project directory.

For real AI projects, this matters because production systems often ingest entire folders instead of one file at a time. You can combine it with `glob` and specific loader classes to process only relevant files.

#### `load()` vs `lazy_load()`

These two methods look similar, but they behave differently:

- `load()`: reads all matching documents into memory at once and returns a list. This is simple and convenient for small to medium datasets.
- `lazy_load()`: streams documents one by one as a generator. This is more memory-efficient and better for large document collections or long-running pipelines.

In AI engineering, the choice matters because large corpora can consume a lot of RAM. If you are processing thousands of files, `lazy_load()` is usually safer and more scalable.

Important considerations:
- Use `glob` to restrict file types and reduce noise
- Keep metadata for source traceability
- Batch processing is great for experimentation, but you should still validate the quality of loaded documents
- For large folders, combine this with chunking and embeddings to build a usable retrieval system


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    path=r'D:\LangChain\langchain-document-loaders-main\books',
    glob='*.pdf',
    loader_cls=PyPDFLoader
)

docs = loader.lazy_load()

for document in docs:
    print(document.metadata)

### 5 WebBase Loader

`WebBaseLoader` lets you pull content from a website into the same LangChain document pipeline. This is helpful for building knowledge systems from public web pages, product pages, docs, or release notes.

In AI engineering, web loading is useful but risky:
- HTML often contains navigation, ads, or irrelevant script text
- Some pages block scraping or change structure frequently
- You must clean and filter content before using it in RAG

A good rule is: load the page content, then clean it, then chunk it, then embed it. Without this step, the model may answer from noisy or outdated content.

In [8]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI()

prompt = PromptTemplate(
    template='Answer the following question \n {question} from the following text - \n {text}',
    input_variables=['question','text']
)

parser = StrOutputParser()

url = 'https://www.flipkart.com/apple-macbook-air-m2-16-gb-256-gb-ssd-macos-sequoia-mc7x4hn-a/p/itmdc5308fa78421'
loader = WebBaseLoader(url)

docs = loader.load()


chain = prompt | model | parser

print(chain.invoke({'question':'What is the prodcut that we are talking about?', 'text':docs[0].page_content}))

USER_AGENT environment variable not set, consider setting it to identify your requests.


The product we are talking about is reCAPTCHA, specifically the version used on Flipkart's website.
